# Hand-wired Agent Loop + Databricks MCP Server

The same agent as `openai_agent_with_mcp.ipynb`, but **without an agent framework**. The whole loop is written out by hand:

1. Ask the MCP server which tools it has and turn them into OpenAI tool definitions.
2. Send the conversation and the tools to the LLM.
3. If the LLM asks for tools → call them on the MCP server, append the results, go back to step 2.
4. If the LLM answers with text → done.

```
┌──────────────────────────┐
│ Question from the user   │
└────────────┬─────────────┘
             ▼
┌──────────────────────────┐
│ 1. Discover tools        │
│    mcp.list_tools()      │
│    → OpenAI tool defs    │
└────────────┬─────────────┘
             ▼
┌──────────────────────────┐
│ 2. Call the LLM          │◄───────────────────────┐
│    messages + tools      │                        │
└────────────┬─────────────┘                        │
             ▼                                      │
     ┌───────────────┐  yes             ┌───────────┴──────────┐
     │ tool_calls?   ├─────────────────►│ 3. Run tools on MCP  │
     └───────┬───────┘                  │    mcp.call_tool()   │
             │ no                       │    append results    │
             ▼                          └──────────────────────┘
┌──────────────────────────┐
│ 4. Final answer (text)   │
└──────────────────────────┘
```

In [ ]:
%pip install -q "databricks-mcp" "openai>=1.0" "databricks-sdk>=0.50.0"

## Configuration

In [ ]:
PROFILE = "resa-prod"                # Databricks CLI profile (only used with LOCAL_MCP)
APP_NAME = "mcp-server-template"    # Databricks App hosting the MCP server
MODEL_NAME = "gpt-5-6-sol"          # any tool-calling serving endpoint
LOCAL_MCP = False                   # True -> use `make run` server on localhost:8000

## Authentication

Databricks Apps only accept OAuth tokens. In the workspace we authenticate as a **service principal** (OAuth M2M), with its client id and secret read from a secret scope. The MCP server then sees the service principal as the signed-in user.

In [ ]:
from databricks.sdk import WorkspaceClient
from databricks_mcp import DatabricksMCPClient

if LOCAL_MCP:
    w = WorkspaceClient(profile=PROFILE)
    mcp_url = "http://localhost:8000/mcp"
else:
    w = WorkspaceClient(
        host=dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiUrl().get(),  # current workspace
        client_id=dbutils.secrets.get("service-principals", "mcp-server-template-user-client-id"),
        client_secret=dbutils.secrets.get("service-principals", "mcp-server-template-user-client-secret"),
        auth_type="oauth-m2m",
    )
    mcp_url = w.apps.get(APP_NAME).url.rstrip("/") + "/mcp"

# Handles OAuth for the request using the WorkspaceClient's credentials
mcp = DatabricksMCPClient(server_url=mcp_url, workspace_client=w)

print("Workspace:", w.config.host)
print("MCP URL:  ", mcp_url)

## LLM client

Databricks serving endpoints are OpenAI-compatible, so a plain `openai` client works.

In [ ]:
llm = w.serving_endpoints.get_open_ai_client()

## Step 1: Discover MCP tools and convert them to OpenAI tools

An MCP tool already comes with a name, a description and a JSON schema for its inputs, which is everything the LLM needs.

In [ ]:
import json

def to_openai_tool(tool) -> dict:
    return {
        "type": "function",
        "function": {
            "name": tool.name,
            "description": tool.description or "",
            "parameters": tool.inputSchema,
        },
    }

mcp_tools = mcp.list_tools()

for tool in mcp_tools:
    print(f"- {tool.name}: {(tool.description or '').strip().splitlines()[0]}")

print("\nAs OpenAI tool definition:")
print(json.dumps(to_openai_tool(mcp_tools[-1]), indent=2))

## Steps 2–4: The agent loop

The whole agent fits in one function. `MAX_STEPS` keeps the loop from running forever.

In [ ]:
SYSTEM_PROMPT = """You are a helpful banking assistant.
Use the available tools to look up the user's account and to forecast savings.
Always state amounts with their currency and explain the assumptions you used."""

MAX_STEPS = 5

def ask(question: str) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]
    tools = [to_openai_tool(t) for t in mcp.list_tools()]

    for _ in range(MAX_STEPS):
        # Step 2: let the LLM decide: answer directly or call tools
        response = llm.chat.completions.create(model=MODEL_NAME, messages=messages, tools=tools)
        msg = response.choices[0].message

        # Step 4: no tool calls means this is the final answer
        if not msg.tool_calls:
            return msg.content

        # Step 3: run every requested tool on the MCP server and feed the results back
        messages.append({
            "role": "assistant",
            "content": msg.content,
            "tool_calls": [
                {"id": tc.id, "type": "function",
                 "function": {"name": tc.function.name, "arguments": tc.function.arguments}}
                for tc in msg.tool_calls
            ],
        })
        for tc in msg.tool_calls:
            args = json.loads(tc.function.arguments or "{}")
            print(f"[tool call]   {tc.function.name}({args})")

            result = mcp.call_tool(tc.function.name, args)
            output = "\n".join(c.text for c in result.content if c.type == "text")
            if result.isError:
                output = f"ERROR: {output}"
            print(f"[tool output] {output[:200]}")

            messages.append({"role": "tool", "tool_call_id": tc.id, "content": output})

    return "Stopped: too many steps."

## Try it

A single tool call:

In [ ]:
print(ask("What is my current account balance?"))

Several tool calls, possibly spread over more than one loop iteration:

In [ ]:
print(ask(
    "If I save 300 per month for 15 years with a 6% annual return, "
    "how much will I have? How much of that is interest?"
))

In [ ]:
print(ask("Compare saving 100 vs 500 per month for 10 years at 5%."))